# Phase Space and Nonlinear Dynamics

## Overview
Phase space (state space) provides a geometric view of dynamical systems. We visualize trajectories, fixed points, stability, and attractors without solving equations explicitly.

## Mathematical Framework

### State Space
The space spanned by all dynamical variables of the system.

For a 2D system:
$$\frac{dx}{dt} = f(x, y)$$
$$\frac{dy}{dt} = g(x, y)$$

Phase space coordinates: (x, y)

### Fixed Points
States where the system doesn't change:
$$f(x^*, y^*) = 0 \quad \text{and} \quad g(x^*, y^*) = 0$$

### Nullclines
Curves where one derivative is zero:
- **x-nullcline**: f(x, y) = 0
- **y-nullcline**: g(x, y) = 0
- Fixed points = intersections of nullclines

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import fsolve
from numpy.linalg import eig

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 10)

## Example 1: FitzHugh-Nagumo Model

Simplified 2D model of neuronal excitability:

$$\frac{dV}{dt} = V - \frac{V^3}{3} - W + I$$
$$\frac{dW}{dt} = \epsilon(V + a - bW)$$

where:
- V: membrane potential (fast variable)
- W: recovery variable (slow variable)
- I: external current
- ε, a, b: parameters

In [ ]:
# FitzHugh-Nagumo parameters
epsilon = 0.08
a = 0.7
b = 0.8
I_ext = 0.5

def fitzhugh_nagumo(state, t, I):
    """FitzHugh-Nagumo ODEs"""
    V, W = state
    dV = V - V**3/3 - W + I
    dW = epsilon * (V + a - b*W)
    return [dV, dW]

print(f"FitzHugh-Nagumo Model Parameters:")
print(f"  ε = {epsilon}")
print(f"  a = {a}")
print(f"  b = {b}")
print(f"  I = {I_ext}")

## Find Fixed Points

In [ ]:
def find_fixed_point(initial_guess, I):
    """Find fixed point using numerical solver"""
    def equations(state):
        return fitzhugh_nagumo(state, 0, I)
    
    fixed_point = fsolve(equations, initial_guess)
    return fixed_point

# Find fixed point
initial_guess = [0, 0]
fp = find_fixed_point(initial_guess, I_ext)
V_fp, W_fp = fp

print(f"\nFixed Point:")
print(f"  V* = {V_fp:.4f}")
print(f"  W* = {W_fp:.4f}")

# Verify it's truly a fixed point
derivatives = fitzhugh_nagumo(fp, 0, I_ext)
print(f"\nDerivatives at fixed point:")
print(f"  dV/dt = {derivatives[0]:.6f}")
print(f"  dW/dt = {derivatives[1]:.6f}")

## Stability Analysis: Jacobian and Eigenvalues

In [ ]:
def jacobian_fhn(V, W, I):
    """Compute Jacobian matrix at (V, W)"""
    # df_V/dV, df_V/dW
    # df_W/dV, df_W/dW
    J = np.array([
        [1 - V**2, -1],
        [epsilon, -epsilon*b]
    ])
    return J

# Compute Jacobian at fixed point
J = jacobian_fhn(V_fp, W_fp, I_ext)
print(f"\nJacobian at fixed point:")
print(J)

# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = eig(J)
print(f"\nEigenvalues:")
for i, lam in enumerate(eigenvalues):
    print(f"  λ{i+1} = {lam:.4f}")
    if lam.real < 0:
        print(f"       → Stable direction")
    elif lam.real > 0:
        print(f"       → Unstable direction")
    else:
        print(f"       → Neutral (bifurcation)")

# Determine stability
if all(eigenvalues.real < 0):
    stability = "STABLE (attractor)"
elif any(eigenvalues.real > 0):
    stability = "UNSTABLE (repeller or saddle)"
else:
    stability = "MARGINAL (bifurcation point)"

print(f"\nFixed point stability: {stability}")

## Phase Portrait: Vector Field and Nullclines

In [ ]:
# Create grid for phase space
V_range = np.linspace(-2.5, 2.5, 25)
W_range = np.linspace(-1, 2.5, 25)
V_grid, W_grid = np.meshgrid(V_range, W_range)

# Compute vector field
dV_grid = V_grid - V_grid**3/3 - W_grid + I_ext
dW_grid = epsilon * (V_grid + a - b*W_grid)

# Nullclines
V_null = np.linspace(-2.5, 2.5, 500)
W_null_V = V_null - V_null**3/3 + I_ext  # V-nullcline: dV/dt = 0
W_null_W = (V_null + a) / b              # W-nullcline: dW/dt = 0

# Plot
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Vector field
ax.quiver(V_grid, W_grid, dV_grid, dW_grid, 
          alpha=0.6, width=0.003, scale=40, color='gray')

# Nullclines
ax.plot(V_null, W_null_V, 'r-', linewidth=2.5, label='V-nullcline (dV/dt=0)')
ax.plot(V_null, W_null_W, 'b-', linewidth=2.5, label='W-nullcline (dW/dt=0)')

# Fixed point
ax.plot(V_fp, W_fp, 'ko', markersize=12, label=f'Fixed point ({stability})')

# Eigenvectors
for i in range(2):
    eigvec = eigenvectors[:, i].real
    # Scale for visibility
    scale = 0.5
    ax.arrow(V_fp, W_fp, scale*eigvec[0], scale*eigvec[1],
             head_width=0.1, head_length=0.1, fc='orange', ec='orange',
             linewidth=2, alpha=0.7)

ax.set_xlabel('V (voltage)', fontsize=13)
ax.set_ylabel('W (recovery)', fontsize=13)
ax.set_title('FitzHugh-Nagumo Phase Portrait', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([-2.5, 2.5])
ax.set_ylim([-1, 2.5])

plt.tight_layout()
plt.savefig('/home/user/phase_portrait_fhn.png', dpi=150, bbox_inches='tight')
plt.show()

## Trajectories from Different Initial Conditions

In [ ]:
# Simulate trajectories from different initial conditions
t = np.linspace(0, 100, 5000)
initial_conditions = [
    [-2, 0],
    [-1, 1],
    [0, 2],
    [1, 1],
    [2, 0.5],
    [-1.5, -0.5]
]

fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Vector field (fainter)
ax.quiver(V_grid, W_grid, dV_grid, dW_grid, 
          alpha=0.3, width=0.002, scale=40, color='lightgray')

# Nullclines
ax.plot(V_null, W_null_V, 'r-', linewidth=2, label='V-nullcline', alpha=0.7)
ax.plot(V_null, W_null_W, 'b-', linewidth=2, label='W-nullcline', alpha=0.7)

# Fixed point
ax.plot(V_fp, W_fp, 'ko', markersize=12, label='Fixed point', zorder=10)

# Trajectories
colors = plt.cm.viridis(np.linspace(0, 1, len(initial_conditions)))
for ic, color in zip(initial_conditions, colors):
    sol = odeint(fitzhugh_nagumo, ic, t, args=(I_ext,))
    V_traj = sol[:, 0]
    W_traj = sol[:, 1]
    
    ax.plot(V_traj, W_traj, color=color, linewidth=2, alpha=0.8)
    ax.plot(ic[0], ic[1], 'o', color=color, markersize=8, markeredgecolor='k', markeredgewidth=1.5)

ax.set_xlabel('V (voltage)', fontsize=13)
ax.set_ylabel('W (recovery)', fontsize=13)
ax.set_title('Phase Space Trajectories from Different Initial Conditions', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim([-2.5, 2.5])
ax.set_ylim([-1, 2.5])

plt.tight_layout()
plt.savefig('/home/user/phase_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

## Example 2: Wilson-Cowan Model

Population dynamics: excitatory (E) and inhibitory (I) populations.

$$\frac{dr_E}{dt} = -r_E + f(w_{EE}r_E - w_{EI}r_I + I)$$
$$\frac{dr_I}{dt} = -r_I + f(w_{IE}r_E - w_{II}r_I + I)$$

where $f(x) = \frac{1}{1 + e^{-sx}}$ (sigmoid)

In [ ]:
# Wilson-Cowan parameters
w_EE = 12.0
w_EI = 4.0
w_IE = 13.0
w_II = 11.0
s = 4.0  # Sigmoid steepness
I_input = 0.0

def sigmoid(x, s=4.0):
    """Sigmoid activation function"""
    return 1.0 / (1.0 + np.exp(-s * x))

def wilson_cowan(state, t, I):
    """Wilson-Cowan population model"""
    r_E, r_I = state
    
    # Input to E population
    h_E = w_EE * r_E - w_EI * r_I + I
    # Input to I population
    h_I = w_IE * r_E - w_II * r_I + I
    
    dr_E = -r_E + sigmoid(h_E, s)
    dr_I = -r_I + sigmoid(h_I, s)
    
    return [dr_E, dr_I]

print(f"Wilson-Cowan Model Parameters:")
print(f"  w_EE = {w_EE}")
print(f"  w_EI = {w_EI}")
print(f"  w_IE = {w_IE}")
print(f"  w_II = {w_II}")
print(f"  Sigmoid steepness: {s}")

## Wilson-Cowan Phase Portrait

In [ ]:
# Find fixed points
def find_wc_fixed_points(I):
    """Find Wilson-Cowan fixed points"""
    def equations(state):
        return wilson_cowan(state, 0, I)
    
    # Try multiple initial guesses
    guesses = [[0.1, 0.1], [0.5, 0.5], [0.9, 0.1]]
    fixed_points = []
    
    for guess in guesses:
        fp = fsolve(equations, guess)
        # Check if valid (rates between 0 and 1)
        if all(0 <= r <= 1 for r in fp):
            # Check if truly a fixed point
            residual = equations(fp)
            if np.linalg.norm(residual) < 1e-6:
                # Check if already found
                is_new = True
                for existing_fp in fixed_points:
                    if np.linalg.norm(fp - existing_fp) < 1e-3:
                        is_new = False
                        break
                if is_new:
                    fixed_points.append(fp)
    
    return fixed_points

fps_wc = find_wc_fixed_points(I_input)
print(f"\nNumber of fixed points found: {len(fps_wc)}")
for i, fp in enumerate(fps_wc):
    print(f"  FP {i+1}: r_E = {fp[0]:.4f}, r_I = {fp[1]:.4f}")

In [ ]:
# Create phase portrait
r_E_range = np.linspace(0, 1, 25)
r_I_range = np.linspace(0, 1, 25)
r_E_grid, r_I_grid = np.meshgrid(r_E_range, r_I_range)

# Compute vector field
dr_E_grid = np.zeros_like(r_E_grid)
dr_I_grid = np.zeros_like(r_I_grid)

for i in range(len(r_E_range)):
    for j in range(len(r_I_range)):
        derivatives = wilson_cowan([r_E_grid[j, i], r_I_grid[j, i]], 0, I_input)
        dr_E_grid[j, i] = derivatives[0]
        dr_I_grid[j, i] = derivatives[1]

# Nullclines (computed numerically)
r_E_null_points = np.linspace(0, 1, 200)
r_I_E_null = []  # E-nullcline
r_I_I_null = []  # I-nullcline

for r_E in r_E_null_points:
    # E-nullcline: find r_I where dr_E/dt = 0
    def e_null_eq(r_I):
        return wilson_cowan([r_E, r_I], 0, I_input)[0]
    try:
        r_I_sol = fsolve(e_null_eq, 0.5)[0]
        if 0 <= r_I_sol <= 1:
            r_I_E_null.append(r_I_sol)
        else:
            r_I_E_null.append(np.nan)
    except:
        r_I_E_null.append(np.nan)
    
    # I-nullcline: find r_I where dr_I/dt = 0
    def i_null_eq(r_I):
        return wilson_cowan([r_E, r_I], 0, I_input)[1]
    try:
        r_I_sol = fsolve(i_null_eq, 0.5)[0]
        if 0 <= r_I_sol <= 1:
            r_I_I_null.append(r_I_sol)
        else:
            r_I_I_null.append(np.nan)
    except:
        r_I_I_null.append(np.nan)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Vector field
ax.quiver(r_E_grid, r_I_grid, dr_E_grid, dr_I_grid,
          alpha=0.6, width=0.003, scale=15, color='gray')

# Nullclines
ax.plot(r_E_null_points, r_I_E_null, 'r-', linewidth=2.5, label='E-nullcline (dr_E/dt=0)')
ax.plot(r_E_null_points, r_I_I_null, 'b-', linewidth=2.5, label='I-nullcline (dr_I/dt=0)')

# Fixed points
for i, fp in enumerate(fps_wc):
    ax.plot(fp[0], fp[1], 'ko', markersize=12, label=f'Fixed point {i+1}')

# Trajectories
t_wc = np.linspace(0, 50, 2000)
ic_wc = [[0.1, 0.1], [0.3, 0.7], [0.7, 0.3], [0.8, 0.8]]
colors_wc = plt.cm.plasma(np.linspace(0, 1, len(ic_wc)))

for ic, color in zip(ic_wc, colors_wc):
    sol = odeint(wilson_cowan, ic, t_wc, args=(I_input,))
    ax.plot(sol[:, 0], sol[:, 1], color=color, linewidth=2, alpha=0.7)
    ax.plot(ic[0], ic[1], 'o', color=color, markersize=8, markeredgecolor='k', markeredgewidth=1.5)

ax.set_xlabel('E population activity', fontsize=13)
ax.set_ylabel('I population activity', fontsize=13)
ax.set_title('Wilson-Cowan Phase Portrait', fontsize=15, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('/home/user/wilson_cowan_phase.png', dpi=150, bbox_inches='tight')
plt.show()

## Time Series from Phase Space

In [ ]:
# Simulate longer time series
t_long = np.linspace(0, 100, 5000)
ic_demo = [0.3, 0.1]
sol_long = odeint(wilson_cowan, ic_demo, t_long, args=(I_input,))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Time series
ax1.plot(t_long, sol_long[:, 0], 'r-', linewidth=2, label='E population')
ax1.plot(t_long, sol_long[:, 1], 'b-', linewidth=2, label='I population')
ax1.set_ylabel('Activity', fontsize=12)
ax1.set_title('Population Dynamics Over Time', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Phase space trajectory
ax2.plot(sol_long[:, 0], sol_long[:, 1], 'g-', linewidth=2, alpha=0.7)
ax2.plot(ic_demo[0], ic_demo[1], 'go', markersize=10, label='Start')
ax2.plot(sol_long[-1, 0], sol_long[-1, 1], 'ro', markersize=10, label='End')
if len(fps_wc) > 0:
    ax2.plot(fps_wc[0][0], fps_wc[0][1], 'ko', markersize=12, label='Fixed point')
ax2.set_xlabel('E population', fontsize=12)
ax2.set_ylabel('I population', fontsize=12)
ax2.set_title('Trajectory in Phase Space', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('/home/user/timeseries_vs_phase.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

1. **Phase Space**: Geometric view of all possible states
2. **Nullclines**: Curves where one variable stops changing
3. **Fixed Points**: Intersections of nullclines
4. **Stability**: Determined by Jacobian eigenvalues
5. **Trajectories**: System evolution visualized as curves
6. **Attractors**: Where trajectories converge (point, cycle, strange)

## Stability Summary

**2D Linear System** $\dot{\mathbf{x}} = J\mathbf{x}$:

| Eigenvalues | Stability | Type |
|-------------|-----------|------|
| Both Re(λ) < 0 | Stable | **Node** (both real) or **Spiral** (complex) |
| Both Re(λ) > 0 | Unstable | **Node** or **Spiral** |
| Mixed signs | Unstable | **Saddle point** |
| Re(λ) = 0 | Marginal | **Center** (pure imaginary) |

## Biological Interpretation

**FitzHugh-Nagumo:**
- Stable fixed point → resting state
- Limit cycle → repetitive firing
- Excitability → large perturbation triggers spike

**Wilson-Cowan:**
- Fixed point → balanced E/I activity
- Oscillations → cortical rhythms
- Multistability → working memory states

## Further Exploration

- Implement bifurcation analysis (vary parameters)
- Study 3D systems (Lorenz attractor, HH neuron)
- Explore strange attractors and chaos
- Apply to experimental data (neural recordings)